In [30]:
import pandas as pd 
import numpy as np 


#verificacao de quantidade e percentual das classes (III)

df_financas = pd.read_csv("financas_data.csv")

contagem = (df_financas["loan_status"].value_counts())
percentual = (df_financas["loan_status"].value_counts(normalize=True) * 100) 

resumo = pd.DataFrame({
    "quantidade": contagem,
    "percentual": percentual.round(2)
    
})
display(resumo)
df_financas.shape


,quantidade,percentual
loan_status,,
0,35000,77.78
1,10000,22.22


(45000, 14)

In [ ]:
#separacao da base para teste e desenvolvimento (IV)

from sklearn.model_selection import train_test_split

x = df_financas.drop(columns=["loan_status"]) #colunas que o programa vai usar para prever menos a target (.drop())
y = df_financas["loan_status"] #coluna alvo isolada

x_dev, x_test, y_dev, y_test = train_test_split(x, y, test_size = 0.2, stratify = y, )

#separacao para pré processsamento 

colunas_numericas = [
    "person_age",
    "person_income",
    "person_emp_exp",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
    "credit_score"
]

colunas_categoricas = [
    "person_gender",
    "person_education",
    "person_home_ownership",
    "loan_intent",
    "previous_loan_defaults_on_file"
]
len(colunas_numericas), len(colunas_categoricas)


#COMENTARIOS 

#SKLEARN - biblioteca
#MODEL_SELECTION - ferramenta do sklearn para separar e avaliar dados
#TRAIN_TEST_SPLIT - dividir/separar os dados

(8, 5)

In [32]:
#pre-processamento e representacao TF-IDF para textos e One-Hot Encoding para variáveis categóricas (V)
#aqui utilizaremos One-Hot pois temos apenas variaveis categoricas e nao de texto 

from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler #coloca variaveis numericas em escala comparaveis
from sklearn.compose import ColumnTransformer

encoder = OneHotEncoder(handle_unknown="ignore")

preprocessador = ColumnTransformer(#objeto
    transformers=[
        ("numericas", StandardScaler(), colunas_numericas),
        ("categoricas", encoder, colunas_categoricas) #aplicar o encoder na colunas_categoricas
        
    ],
    remainder="passthrough" #nao passe pelas colunas que nao foram mencionadas
)

x_dev_preparado = preprocessador.fit_transform(x_dev) #fit_transform == aprenda as categorias e transforme em dados 
x_test_preparado = preprocessador.transform(x_test)

x_dev_preparado.shape
x_test_preparado.shape
preprocessador.get_feature_names_out()

array(['numericas__person_age', 'numericas__person_income',
       'numericas__person_emp_exp', 'numericas__loan_amnt',
       'numericas__loan_int_rate', 'numericas__loan_percent_income',
       'numericas__cb_person_cred_hist_length', 'numericas__credit_score',
       'categoricas__person_gender_female',
       'categoricas__person_gender_male',
       'categoricas__person_education_Associate',
       'categoricas__person_education_Bachelor',
       'categoricas__person_education_Doctorate',
       'categoricas__person_education_High School',
       'categoricas__person_education_Master',
       'categoricas__person_home_ownership_MORTGAGE',
       'categoricas__person_home_ownership_OTHER',
       'categoricas__person_home_ownership_OWN',
       'categoricas__person_home_ownership_RENT',
       'categoricas__loan_intent_DEBTCONSOLIDATION',
       'categoricas__loan_intent_EDUCATION',
       'categoricas__loan_intent_HOMEIMPROVEMENT',
       'categoricas__loan_intent_MEDICAL',
      

In [ ]:
#montando pipeline e fazendo a validacao cruzada

from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold #StratifiedKFold - fazer validacao cruzada

validacao_cruzada = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Modelo 1: Regressão Logística
pipeline_logreg = Pipeline([
    ("preprocessamento", preprocessador),
    ("modelo", LogisticRegression(max_iter=2000, random_state=42))
])
parametros_logreg = {"modelo__C": [0.01, 0.1, 1, 10]}

# Modelo 2: Random Forest
pipeline_rf = Pipeline([
    ("preprocessamento", preprocessador),
    ("modelo", RandomForestClassifier(random_state=42, n_jobs=1))
])
parametros_rf = {
    "modelo__n_estimators": [100, 200],
    "modelo__max_depth": [5, 10, None]
}

# Modelo 3: Gradient Boosting
pipeline_gb = Pipeline([
    ("preprocessamento", preprocessador),
    ("modelo", GradientBoostingClassifier(random_state=42))
])
parametros_gb = {
    "modelo__n_estimators": [100, 200],
    "modelo__learning_rate": [0.05, 0.1],
    "modelo__max_depth": [2, 3]
}

print(pipeline_logreg)

Pipeline(steps=[('preprocessamento',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numericas', StandardScaler(),
                                                  ['person_age',
                                                   'person_income',
                                                   'person_emp_exp',
                                                   'loan_amnt', 'loan_int_rate',
                                                   'loan_percent_income',
                                                   'cb_person_cred_hist_length',
                                                   'credit_score']),
                                                 ('categoricas',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['person_gender',
                                                   'person_education',
                     

In [ ]:
#Regressao Logostica (VI) relatorio
busca_logreg = RandomizedSearchCV(
    estimator=pipeline_logreg,
    param_distributions=parametros_logreg,
    cv=validacao_cruzada,
    scoring="average_precision",
    n_iter=4,
    random_state=42,
    n_jobs=-1
)

busca_logreg.fit(x_dev, y_dev)

print("Melhores parâmetros:")
print(busca_logreg.best_params_)

print("\nMelhor Average precision na validação (RL):")
print(busca_logreg.best_score_)

#Random Forest
busca_rf = RandomizedSearchCV(
    estimator=pipeline_rf,
    param_distributions=parametros_rf,
    cv=validacao_cruzada,
    scoring="average_precision",
    n_iter=6,
    random_state=42,
    n_jobs=-1
)

busca_rf.fit(x_dev, y_dev)

print("Melhores parâmetros:")
print(busca_rf.best_params_)

print("\nMelhor Average Precision na validação:")
print(busca_rf.best_score_)

#Grandient Boosting
busca_gb = RandomizedSearchCV(
    estimator=pipeline_gb,
    param_distributions=parametros_gb,
    cv=validacao_cruzada,
    scoring="average_precision",
    n_iter=8,
    random_state=42,
    n_jobs=-1
)

busca_gb.fit(x_dev, y_dev)

print("Melhores parâmetros:")
print(busca_gb.best_params_)

print("\nMelhor Average Precision na validação ():")
print(busca_gb.best_score_)

Melhores parâmetros:
{'modelo__C': 0.1}

Melhor Average precision na validação (RL):
0.8535215971841087
Melhores parâmetros:
{'modelo__n_estimators': 200, 'modelo__max_depth': None}

Melhor Average Precision na validação:
0.9256615094505822
Melhores parâmetros:
{'modelo__n_estimators': 200, 'modelo__max_depth': 3, 'modelo__learning_rate': 0.1}

Melhor Average Precision na validação ():
0.9296337954071714


In [ ]:
#(II) e (III)programa  
# Melhores modelos encontrados na validação
modelo_logreg = busca_logreg.best_estimator_
modelo_rf = busca_rf.best_estimator_
modelo_gb = busca_gb.best_estimator_

# Probabilidade estimada da classe positiva (1) para cada instância do teste
prob_classe1_logreg = modelo_logreg.predict_proba(x_test)[:, 1]
prob_classe1_rf = modelo_rf.predict_proba(x_test)[:, 1]
prob_classe1_gb = modelo_gb.predict_proba(x_test)[:, 1]


resultados_prob = pd.DataFrame({
    "Rótulo real": y_test.values,
    "Prob. Logística": prob_classe1_logreg * 100,
    "Prob. Random Forest": prob_classe1_rf * 100,
    "Prob. Gradient Boosting": prob_classe1_gb * 100
})

print(resultados_prob.head(10).round(2))
y_real = y_test
print(y_real[:10])

   Rótulo real  Prob. Logística  Prob. Random Forest  Prob. Gradient Boosting
0            0             0.02                  0.5                     0.16
1            0             0.11                  0.0                     0.02
2            0             0.11                  1.0                     0.07
3            0             0.28                  0.0                     0.16
4            0             0.10                  0.0                     0.05
5            1            40.69                 78.0                    71.98
6            1            91.63                 60.5                    78.97
7            0             0.01                  0.0                     0.03
8            0            33.96                 45.5                    45.11
9            1            90.25                 97.0                    98.14
33723    0
22356    0
17882    0
7047     0
34901    0
13468    1
43703    1
39103    0
18084    0
43684    1
Name: loan_status, dtype: int64
